### Importando Bibliotecas

In [10]:
import sys
!{sys.executable} -m pip install scikit-learn pandas numpy
import pandas as pd
import numpy as np
import time
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from scipy.optimize import minimize


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1. Separando em Treino e Teste

In [13]:
# 1. Carregando os dados tratados
# Seguindo a estrutura de pastas do seu GitHub
df = pd.read_csv("../Tratamento_de_Dados/dataset_tratado.csv")

# Removendo coluna residual se existir, como no seu código de SVM
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', inplace=True, axis=1)

X = df.drop('Group', axis=1)
y = df['Group']

In [14]:
df.head()

,Age,CDR,SES,nWBV,Visit,MMSE,ASF,eTIV,Group,MR Delay
0,87.0,0.0,2.0,0.696106,1.0,27.0,0.883440,1986.550000,0.0,0.0
1,88.0,0.0,2.0,0.681062,2.0,30.0,0.875539,2004.479526,0.0,457.0
2,75.0,0.5,1.8,0.736336,1.0,23.0,1.045710,1678.290000,1.0,0.0
3,76.0,0.5,1.6,0.713402,2.0,28.0,1.010000,1737.620000,1.0,560.0
4,80.0,0.5,2.6,0.701236,3.0,22.0,1.033623,1697.911134,1.0,1895.0


In [15]:
# 2. Divisão entre Treino e Teste (80/20)
# Mantendo o RANDOM_STATE=42 para consistência com os outros modelos do grupo
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### 1.1 Treinando o Modelo

In [16]:
def ModelagemRL(X_train, y_train, X_test, Solver):
    """
    Função adaptada para testar diferentes solvers de otimização na Regressão Logística.
    """
    pipeline = Pipeline([
        ('scaler', StandardScaler()),        # Normalização essencial para solvers de gradiente 
        ('pca', PCA(n_components=0.95)),     # Redução de dimensionalidade conforme o artigo 
        ('modelo', LogisticRegression(solver=Solver, max_iter=1000, tol=1e-5))   
    ])
    
    start_time = time.perf_counter()
    pipeline.fit(X_train, y_train)
    end_time = time.perf_counter()
    
    previsoes = pipeline.predict(X_test)
    tempo_execucao = end_time - start_time
    
    return previsoes, tempo_execucao

## 2. O Problema de Otimização

Diferente de modelos de árvore, a **Regressão Logística (LR)** é fundamentalmente um problema de otimização contínua.Nosso objetivo é encontrar o vetor de parâmetros $w$ que maximize a probabilidade de os dados observados pertencerem às suas classes reais, o que é feito através da minimização de uma função de perda.

### 2.1 Função de Hipótese e Sigmoide
Para mapear qualquer valor real em um intervalo de probabilidade $[0, 1]$, utilizamos a função logística (sigmoide):

$$P(Y) = \frac{1}{1+e^{-(b_{0}+b_{1}x_{1}+b_{2}x_{2}+...+b_{n}x_{n})}}$$

Onde:
* $b_0, b_1, \dots, b_n$ são os coeficientes (pesos) que o modelo precisa aprender via otimização.
* $x_1, x_2, \dots, x_n$ representam as features do dataset OASIS, como Idade, nWBV, MMSE, entre outras.
* A saída representa se o paciente possui Alzheimer (codificado como 1) ou não (codificado como 0).



### 2.2 Função de Custo: Log-Loss
Em Otimização Não-Linear, resolvemos este problema minimizando a **Negative Log-Likelihood** (ou Log-Loss). Esta função mede a discrepância entre as predições e os rótulos reais:

$$J(w) = -\frac{1}{N} \sum_{i=1}^{N} [y_i \log(h_w(x_i)) + (1 - y_i) \log(1 - h_w(x_i))]$$

### 2.3 Convexidade e Solvers
Um ponto crucial para a disciplina é que a função de custo da Regressão Logística é **convexa**. Isso garante que:
* Não existem mínimos locais; qualquer mínimo encontrado pelos algoritmos é o **mínimo global**.
* Podemos utilizar métodos de otimização eficientes como **L-BFGS**, **Newton-CG** ou **Liblinear** para convergência rápida.



No artigo base, a eficiência deste processo de otimização permitiu que a Regressão Logística atingisse **96% de acurácia, precisão, sensibilidade e F1-score** , demonstrando ser um preditor tão robusto quanto o SVM e o Random Forest para a doença de Alzheimer.

In [11]:
class LR_Optimizer:
    def __init__(self, tol=1e-5, maxiter=1000):
        self.tol = tol
        self.maxiter = maxiter

    def sigmoid(self, z):
        # A função sigmoide mapeia os valores para o intervalo [0, 1] 
        return 1 / (1 + np.exp(-np.clip(z, -250, 250)))

    def loss(self, w, X, y):
        # Implementação da Negative Log-Likelihood (Log-Loss)
        n = len(y)
        h = self.sigmoid(X @ w)
        # Adicionamos um pequeno epsilon para evitar log(0)
        epsilon = 1e-15
        return - (1/n) * np.sum(y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon))

    def gradient(self, w, X, y):
        # O gradiente da Regressão Logística: (1/n) * X.T * (h - y)
        n = len(y)
        h = self.sigmoid(X @ w)
        return (1/n) * X.T @ (h - y)

    def hessiana(self, w, X, y):
        # A Hessiana é necessária para o método de Newton-CG
        n = len(y)
        h = self.sigmoid(X @ w)
        # Matriz diagonal R onde R_ii = h_i * (1 - h_i)
        R = np.diag(h * (1 - h))
        return (1/n) * X.T @ R @ X

    def otimizar(self, X, y, w0, methods):
        max_iter_options = {
            'CG':        {'maxiter': self.maxiter},
            'BFGS':      {'maxiter': self.maxiter},
            'Newton-CG': {'maxiter': self.maxiter},
            'L-BFGS-B':  {'maxiter': self.maxiter}
        }

        results = {}
        for method in methods:
            iters = []
            start = time.perf_counter()

            res = minimize(
                fun=self.loss,
                x0=w0,
                jac=self.gradient,
                hess=self.hessiana if method == 'Newton-CG' else None,
                args=(X, y),
                method=method,
                callback=lambda w: iters.append(self.loss(w, X, y)),
                tol=self.tol,
                options=max_iter_options[method]
            )

            elapsed = time.perf_counter() - start
            results[method] = {
                'time':      elapsed,
                'loss':      res.fun,
                'iters':     len(iters),
                'converged': res.success,
                'w':         res.x
            }
            print(f"{method:12s} | tempo: {elapsed:.4f}s | loss: {res.fun:.4f} | iterações: {len(iters)} | convergiu: {res.success}")

        return results

    def predict(self, X, w):
        # Se probabilidade > 0.5, classe 1 (Demented), senão 0 [cite: 135, 136]
        prob = self.sigmoid(X @ w)
        return (prob >= 0.5).astype(int)

    def avaliar(self, X_test, y_test, results):
        for method, data in results.items():
            y_pred = self.predict(X_test, data['w'])
            f1  = f1_score(y_test, y_pred)
            acc = accuracy_score(y_test, y_pred)
            print(f"  {method:12s} | f1: {f1:.4f} | acc: {acc:.4f} | convergiu: {data['converged']}")

In [18]:
X_train_np = np.asarray(X_train)
y_train_np = np.asarray(y_train)
X_test_np = np.asarray(X_test)
y_test_np = np.asarray(y_test)

# Inicialização com zeros (vetor com o mesmo número de colunas de X)
w0 = np.zeros(X_train_np.shape[1])
methods = ['CG', 'BFGS', 'Newton-CG', 'L-BFGS-B']

print("--- Otimização da Regressão Logística ---")
lr_opt = LR_Optimizer(maxiter=1000)

# Note que agora passamos as versões _np
resultados_rl = lr_opt.otimizar(X_train_np, y_train_np, w0, methods)
lr_opt.avaliar(X_test_np, y_test_np, resultados_rl)

--- Otimização da Regressão Logística ---
CG           | tempo: 0.3272s | loss: 0.1266 | iterações: 1000 | convergiu: False
BFGS         | tempo: 0.0150s | loss: 0.0783 | iterações: 93 | convergiu: True
Newton-CG    | tempo: 0.0070s | loss: 0.6473 | iterações: 3 | convergiu: True
L-BFGS-B     | tempo: 0.0320s | loss: 0.4777 | iterações: 35 | convergiu: True
  CG           | f1: 0.9538 | acc: 0.9600 | convergiu: False
  BFGS         | f1: 0.9538 | acc: 0.9600 | convergiu: True
  Newton-CG    | f1: 0.0000 | acc: 0.5733 | convergiu: True
  L-BFGS-B     | f1: 0.5926 | acc: 0.7067 | convergiu: True
